In [1]:
from pyspark.sql import functions as F

HDFS_BASE = "hdfs:///user/vagrant"

LISTINGS_WITH_POLICE = f"{HDFS_BASE}/bigdata/analytical/listings_with_police_stats"

NEIGH_OUT = f"{HDFS_BASE}/bigdata/analytical/neighbourhood_stats"
CITY_OUT  = f"{HDFS_BASE}/bigdata/analytical/city_comparisons"

print("INPUT :", LISTINGS_WITH_POLICE)
print("NEIGH :", NEIGH_OUT)
print("CITY  :", CITY_OUT)

INPUT : hdfs:///user/vagrant/bigdata/analytical/listings_with_police_stats
NEIGH : hdfs:///user/vagrant/bigdata/analytical/neighbourhood_stats
CITY  : hdfs:///user/vagrant/bigdata/analytical/city_comparisons


In [2]:
df = spark.read.parquet(LISTINGS_WITH_POLICE)

print("Rows:", df.count())
df.groupBy("city_code").count().show()

df.printSchema()
df.show(5, truncate=80)

Rows: 400


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|  200|
|      NYC|  200|
+---------+-----+

root
 |-- id: string (nullable = true)
 |-- city_code: string (nullable = true)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- latitude_d: double (nullable = true)
 |-- longitude_d: double (nullable = true)
 |-- geohash5: string (nullable = true)
 |-- price_num: double (nullable = true)
 |-- room_type: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bedrooms: float (nullable = true)
 |-- beds: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: float (nullable = true)
 |-- availability_365: integer (nullable = true)
 |-- host_is_superhost: boolean (nullable = true)
 |-- host_response_rate: float (nullable = true)
 |-- host_acceptance_rate: float (nullable = true)
 |-- crime_events_total: long (nullable = true)
 |-- crime_events_distinct: long (n

In [3]:
df.select(
    F.sum(F.col("city_code").isNull().cast("int")).alias("city_nulls"),
    F.sum(F.col("neighbourhood_cleansed").isNull().cast("int")).alias("neigh_nulls"),
    F.sum(F.col("price_num").isNull().cast("int")).alias("price_nulls"),
    F.sum(F.col("crime_events_total").isNull().cast("int")).alias("crime_nulls"),
).show()

+----------+-----------+-----------+-----------+
|city_nulls|neigh_nulls|price_nulls|crime_nulls|
+----------+-----------+-----------+-----------+
|         0|          0|        148|          0|
+----------+-----------+-----------+-----------+



In [ ]:
neigh_stats = (
    df.groupBy("city_code", "neighbourhood_cleansed")
      .agg(
          F.count("*").alias("listing_count"),
          F.avg("price_num").alias("avg_price"),
          F.expr("percentile_approx(price_num, 0.5)").alias("median_price"),
          F.avg("review_scores_rating").alias("avg_rating"),
          F.expr("percentile_approx(review_scores_rating, 0.5)").alias("median_rating"),
          F.avg("crime_events_total").alias("avg_crimes_nearby"),
          F.expr("percentile_approx(crime_events_total, 0.5)").alias("median_crimes_nearby"),
          F.avg("safety_index").alias("avg_safety_index"),
          F.expr("percentile_approx(safety_index, 0.5)").alias("median_safety_index"),
          
          F.sum((F.col("crime_bucket") == "0").cast("int")).alias("bucket_0_cnt"),
          F.sum((F.col("crime_bucket") == "1-5").cast("int")).alias("bucket_1_5_cnt"),
          F.sum((F.col("crime_bucket") == "6-20").cast("int")).alias("bucket_6_20_cnt"),
          F.sum((F.col("crime_bucket") == "21+").cast("int")).alias("bucket_21p_cnt"),
      )
)

neigh_stats.orderBy(F.col("listing_count").desc()).show(10, truncate=False)

[Stage 17:===================================================>  (189 + 3) / 200]

+---------+----------------------+-------------+------------------+------------+-----------------+-------------+------------------+--------------------+---------------------+---------------------+------------+--------------+---------------+--------------+
|city_code|neighbourhood_cleansed|listing_count|avg_price         |median_price|avg_rating       |median_rating|avg_crimes_nearby |median_crimes_nearby|avg_safety_index     |median_safety_index  |bucket_0_cnt|bucket_1_5_cnt|bucket_6_20_cnt|bucket_21p_cnt|
+---------+----------------------+-------------+------------------+------------+-----------------+-------------+------------------+--------------------+---------------------+---------------------+------------+--------------+---------------+--------------+
|LON      |Hackney               |23           |156.52941176470588|155.0       |4.793333326067243|4.86         |2744.0434782608695|3322                |3.961335408231345E-4 |3.009328919650918E-4 |0           |0             |0       

In [5]:
neigh_stats = neigh_stats.withColumn(
    "crimes_per_100_listings",
    (F.col("avg_crimes_nearby") * 100.0 / F.col("listing_count"))
)

neigh_stats.select(
    "city_code","neighbourhood_cleansed","listing_count",
    "avg_price","median_price","avg_crimes_nearby","crimes_per_100_listings","avg_safety_index"
).orderBy("city_code", F.col("avg_crimes_nearby").desc()).show(10, truncate=False)

[Stage 19:=============================================>        (169 + 2) / 200]

+---------+----------------------+-------------+------------------+------------+------------------+-----------------------+---------------------+
|city_code|neighbourhood_cleansed|listing_count|avg_price         |median_price|avg_crimes_nearby |crimes_per_100_listings|avg_safety_index     |
+---------+----------------------+-------------+------------------+------------+------------------+-----------------------+---------------------+
|LON      |Camden                |13           |187.0             |100.0       |4130.615384615385 |31773.964497041423     |3.799856134400684E-4 |
|LON      |Islington             |12           |118.125           |120.0       |3246.1666666666665|27051.388888888887     |3.682312220847351E-4 |
|LON      |Tower Hamlets         |11           |133.6             |129.0       |2982.4545454545455|27113.223140495866     |3.501053633750847E-4 |
|LON      |Westminster           |9            |355.0             |350.0       |2817.0            |31300.0                |3

In [6]:
(neigh_stats
 .write
 .mode("overwrite")
 .parquet(NEIGH_OUT)
)

print("Saved neighbourhood_stats to:", NEIGH_OUT)

26/01/13 02:23:36 WARN hdfs.DFSClient: Caught exception           (3 + 2) / 200]
java.lang.InterruptedException
	at java.lang.Object.wait(Native Method)
	at java.lang.Thread.join(Thread.java:1252)
	at java.lang.Thread.join(Thread.java:1326)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.closeResponder(DFSOutputStream.java:716)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.endBlock(DFSOutputStream.java:476)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.run(DFSOutputStream.java:652)
                                                                                

Saved neighbourhood_stats to: hdfs:///user/vagrant/bigdata/analytical/neighbourhood_stats


In [ ]:
city_stats = (
    df.groupBy("city_code")
      .agg(
          F.count("*").alias("listing_count"),
          F.avg("price_num").alias("avg_price"),
          F.expr("percentile_approx(price_num, 0.5)").alias("median_price"),
          F.avg("review_scores_rating").alias("avg_rating"),
          F.expr("percentile_approx(review_scores_rating, 0.5)").alias("median_rating"),
          F.avg("crime_events_total").alias("avg_crimes_nearby"),
          F.expr("percentile_approx(crime_events_total, 0.5)").alias("median_crimes_nearby"),
          F.avg("safety_index").alias("avg_safety_index"),
          F.expr("percentile_approx(safety_index, 0.5)").alias("median_safety_index"),
          
          F.sum((F.col("crime_bucket") == "0").cast("int")).alias("bucket_0_cnt"),
          F.sum((F.col("crime_bucket") == "1-5").cast("int")).alias("bucket_1_5_cnt"),
          F.sum((F.col("crime_bucket") == "6-20").cast("int")).alias("bucket_6_20_cnt"),
          F.sum((F.col("crime_bucket") == "21+").cast("int")).alias("bucket_21p_cnt"),
      )
)

city_stats = city_stats.withColumn(
    "bucket_0_pct", F.col("bucket_0_cnt") / F.col("listing_count")
).withColumn(
    "bucket_1_5_pct", F.col("bucket_1_5_cnt") / F.col("listing_count")
).withColumn(
    "bucket_6_20_pct", F.col("bucket_6_20_cnt") / F.col("listing_count")
).withColumn(
    "bucket_21p_pct", F.col("bucket_21p_cnt") / F.col("listing_count")
)

city_stats.show(truncate=False)

+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+------------+--------------+---------------+--------------+------------+--------------+---------------+--------------+
|city_code|listing_count|avg_price         |median_price|avg_rating       |median_rating|avg_crimes_nearby|median_crimes_nearby|avg_safety_index    |median_safety_index |bucket_0_cnt|bucket_1_5_cnt|bucket_6_20_cnt|bucket_21p_cnt|bucket_0_pct|bucket_1_5_pct|bucket_6_20_pct|bucket_21p_pct|
+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+------------+--------------+---------------+--------------+------------+--------------+---------------+--------------+
|LON      |200          |153.23134328358208|105.0       |4.745057465016157|4.8          |1954.31          |1746                |9.308

In [ ]:
corrs = (
    df.groupBy("city_code")
      .agg(
          F.corr("price_num", "crime_events_total").alias("corr_price_vs_crime"),
          F.corr("review_scores_rating", "crime_events_total").alias("corr_rating_vs_crime"),
          F.corr("price_num", "safety_index").alias("corr_price_vs_safety"),
      )
)

corrs.show(truncate=False)

city_stats = city_stats.join(corrs, on="city_code", how="left")
city_stats.show(truncate=False)

+---------+--------------------+--------------------+--------------------+
|city_code|corr_price_vs_crime |corr_rating_vs_crime|corr_price_vs_safety|
+---------+--------------------+--------------------+--------------------+
|LON      |0.1734342850148531  |0.1595360391448731  |-0.07811185289299533|
|NYC      |-0.09940260392593177|0.05342207267597718 |0.08426011617829839 |
+---------+--------------------+--------------------+--------------------+



+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+------------+--------------+---------------+--------------+------------+--------------+---------------+--------------+--------------------+--------------------+--------------------+
|city_code|listing_count|avg_price         |median_price|avg_rating       |median_rating|avg_crimes_nearby|median_crimes_nearby|avg_safety_index    |median_safety_index |bucket_0_cnt|bucket_1_5_cnt|bucket_6_20_cnt|bucket_21p_cnt|bucket_0_pct|bucket_1_5_pct|bucket_6_20_pct|bucket_21p_pct|corr_price_vs_crime |corr_rating_vs_crime|corr_price_vs_safety|
+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+------------+--------------+---------------+--------------+------------+--------------+---------------+--------------+--------

In [9]:
(city_stats
 .write
 .mode("overwrite")
 .parquet(CITY_OUT)
)

print("Saved city_comparisons to:", CITY_OUT)

[Stage 57:===================================================>  (190 + 2) / 200]

Saved city_comparisons to: hdfs:///user/vagrant/bigdata/analytical/city_comparisons


In [10]:
neigh_check = spark.read.parquet(NEIGH_OUT)
city_check  = spark.read.parquet(CITY_OUT)

print("Neighbourhood rows:", neigh_check.count())
neigh_check.groupBy("city_code").count().show()

print("City rows:", city_check.count())
city_check.show(truncate=False)

[Stage 60:===================>                                      (1 + 2) / 3]

Neighbourhood rows: 79


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|   27|
|      NYC|   52|
+---------+-----+

City rows: 2
+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+------------+--------------+---------------+--------------+------------+--------------+---------------+--------------+--------------------+--------------------+--------------------+
|city_code|listing_count|avg_price         |median_price|avg_rating       |median_rating|avg_crimes_nearby|median_crimes_nearby|avg_safety_index    |median_safety_index |bucket_0_cnt|bucket_1_5_cnt|bucket_6_20_cnt|bucket_21p_cnt|bucket_0_pct|bucket_1_5_pct|bucket_6_20_pct|bucket_21p_pct|corr_price_vs_crime |corr_rating_vs_crime|corr_price_vs_safety|
+---------+-------------+------------------+------------+-----------------+-------------+-----------------+--------------------+--------------------+--------------------+----